# Comparativa con AutoML — PyCaret

> **Entorno separado.** PyCaret fija versiones de scikit-learn y pandas que
> entran en conflicto con TensorFlow y XGBoost, así que necesita su propio
> entorno y su propio kernel:
>
> ```
> conda create -n pycaret python=3.11 -y
> conda activate pycaret
> pip install pycaret ipykernel matplotlib
> python -m ipykernel install --user --name pycaret --display-name "Python (pycaret)"
> ```

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from pycaret.classification import *

df = pd.read_csv("data/raw/dataset_practica_final.csv")

## Configuración del experimento

A PyCaret se le pasa el **CSV crudo**, no los datos ya preprocesados: su
`setup()` construye internamente su propio pipeline de imputación y
codificación.

`ignore_features` es la línea más importante de todo el notebook. Sin ella,
todos los modelos dan AUC 1,0 porque `reservation_status` reproduce el target
con correspondencia 1 a 1, y la tabla entera no vale nada. PyCaret no emite
ningún aviso al respecto.

In [ ]:
s = setup(
    data=df,
    target="is_canceled",
    ignore_features=["reservation_status", "reservation_status_date"],
    train_size=0.8,
    fold=5,
    session_id=42,
    fix_imbalance=False,
    normalize=False,
)

**Conviene revisar la tabla que imprime `setup()`**: indica qué columnas ha
tratado como categóricas. El punto a vigilar es `agent`, que viene como número
pero es un identificador (el agente 240 no es "más" que el 9). Si lo ha tomado
como numérica, se corrige con `categorical_features=["agent"]`.

## Comparativa inicial de algoritmos

`compare_models()` entrena una quincena de algoritmos con validación cruzada de
5 folds. La columna **Std** es tan informativa como la media: si dos modelos se
diferencian menos que su desviación, la diferencia no es significativa.

In [ ]:
# Comparativa inicial de todos los algoritmos disponibles
comparacion_inicial = compare_models(sort="AUC", n_select=3)

## Modelos por defecto de PyCaret

In [ ]:
# Modelos por defecto de PyCaret
modelo_rf_default = create_model("rf")

In [ ]:
modelo_xgb_default = create_model("xgboost")

## Modelos con los hiperparámetros elegidos

Son los que seleccioné en el notebook de sklearn tras el proceso de
entrenamientos previos.

In [ ]:
# Modelos con los hiperparámetros elegidos por nosotros
modelo_rf_entrenado  = create_model("rf",n_estimators=300,min_samples_leaf=2,max_depth=None,n_jobs=-1,random_state=42)

In [ ]:
modelo_xgb_entrenado = create_model("xgboost",n_estimators=300,learning_rate=0.1,max_depth=6,n_jobs=-1,random_state=42)

## Búsqueda automática de hiperparámetros

`tune_model()` hace una búsqueda aleatoria con validación cruzada. **No parte de
los hiperparámetros del modelo que se le pasa**: lo único que hereda es el
algoritmo, y el resto lo busca dentro de su propia rejilla.

In [ ]:
rf_tuned = tune_model(modelo_rf_default, optimize="AUC", n_iter=20)

In [ ]:
xgb_tuned = tune_model(modelo_xgb_default, optimize="AUC", n_iter=20)

In [ ]:
print("Random Forest ajustado:", rf_tuned)
print("XGBoost ajustado:", xgb_tuned)

Compararlos con los elegidos a mano es el contraste más interesante: si la
búsqueda converge a un `min_samples_leaf` bajo, respalda con datos la decisión
que tomé por razonamiento.

## Gráficos

In [ ]:
plot_model(rf_tuned, plot="confusion_matrix")

In [ ]:
plot_model(rf_tuned, plot="auc")

In [ ]:
plot_model(rf_tuned, plot="pr")

In [ ]:
plot_model(rf_tuned, plot="feature")

## Comparación total

Se enfrentan los modelos por defecto, los que entrené con mis hiperparámetros y
los que encontró la búsqueda automática.

In [ ]:
#Comparacion total con los modelos por defectos y los entrenados por nosotros
comparacion = compare_models(include=[
    modelo_rf_default,
    modelo_rf_entrenado,
    rf_tuned,
    modelo_xgb_default,
    modelo_xgb_entrenado,
    xgb_tuned,
], sort="AUC")

In [ ]:
resultados = pull()
resultados